In [1]:
from pathlib import Path
import pandas as pd 
import numpy as np
import pingouin as pg
from dowhy import CausalModel
import statsmodels.api as sm
from scipy.stats import t

import warnings
warnings.filterwarnings('ignore')

import modules

pd.set_option('display.max_columns', None)

### Causal Discovery based on morning rides in 2024

In [2]:
mode = 'Train' # 'Train' or 'Test'

vol_rts_df = modules.data_prep(mode)

vol_rts_df = vol_rts_df.loc[vol_rts_df['time_of_day'] == 2].reset_index(drop = True) # Morning rides only

print(len(vol_rts_df))
vol_rts_df.head()

98675


,date_of_trip,day_of_week,time_of_day,PULocationID,DOLocationID,uber_vol,uber_trip_dist,uber_mph,uber_trip_dur,uber_wait_dur,uber_wait_ratio,uber_fare_per_mile,uber_pay_per_mile,lyft_vol,lyft_trip_dist,lyft_mph,lyft_trip_dur,lyft_wait_dur,lyft_wait_ratio,lyft_fare_per_mile,lyft_pay_per_mile,tot_vol,uber_pay,uber_fare,lyft_pay,lyft_fare,uber_wait,lyft_wait,lyft_share,month_of_trip,sunday_date
0,2024-01-01,0,2,7,7,41,0.892927,11.505134,4.982439,5.090488,1.085918,12.954292,4.188248,28,1.192714,11.224988,6.850714,2.287500,0.402386,10.578263,6.784952,69,4.188248,12.954292,6.784952,10.578263,5.090488,2.287500,0.405797,2024-01,2023-12-30
1,2024-01-01,0,2,7,129,18,2.632222,12.911368,13.285000,5.641111,0.473235,6.689509,3.362486,19,2.420526,14.729228,10.046316,2.914737,0.315553,5.875484,3.917398,37,3.362486,6.689509,3.917398,5.875484,5.641111,2.914737,0.513514,2024-01,2023-12-30
2,2024-01-01,0,2,7,179,16,1.315000,11.499533,7.094375,5.096250,0.787224,9.592964,3.641561,15,1.124600,11.713507,5.872000,1.821333,0.342103,9.307941,6.381631,31,3.641561,9.592964,6.381631,9.307941,5.096250,1.821333,0.483871,2024-01,2023-12-30
3,2024-01-01,0,2,7,223,34,1.883529,12.813989,8.185882,5.561471,0.833741,8.992748,2.841334,16,1.331125,10.645015,7.882500,2.163750,0.301799,8.697696,5.717971,50,2.841334,8.992748,5.717971,8.697696,5.561471,2.163750,0.320000,2024-01,2023-12-30
4,2024-01-01,0,2,7,265,14,23.998571,34.367627,40.161429,7.330714,0.194777,5.403085,2.305692,12,20.411333,35.421365,34.518333,3.275000,0.100611,4.288168,2.389737,26,2.305692,5.403085,2.389737,4.288168,7.330714,3.275000,0.461538,2024-01,2023-12-30


In [3]:
res = modules.pc_three_vars(
    vol_rts_df,
    ["uber_fare", "uber_pay", "uber_wait"],
    alpha=0.05,
    max_samples=1000
)

=== PC on 3 variables ===
Nodes: ['uber_fare', 'uber_pay', 'uber_wait']
Rows used (after dropna + trim): 1000

Marginal correlation tests:
uber_fare -- uber_pay: r=0.874, p=4.22e-315
uber_fare -- uber_wait: r=0.154, p=9.95e-07
uber_pay -- uber_wait: r=0.021, p=0.515

Conditional (partial) correlation tests:
uber_fare -- uber_pay | uber_wait: r=0.882, p=0
uber_fare -- uber_wait | uber_pay: r=0.280, p=1.94e-19
uber_pay -- uber_wait | uber_fare: r=-0.237, p=2.89e-14

Learned structure:
uber_pay -> uber_fare
uber_wait -> uber_fare


In [4]:
res = modules.pc_three_vars(
    vol_rts_df,
    ["lyft_fare", "lyft_pay", "lyft_wait"],
    alpha=0.05,
    max_samples=1000
)

=== PC on 3 variables ===
Nodes: ['lyft_fare', 'lyft_pay', 'lyft_wait']
Rows used (after dropna + trim): 1000

Marginal correlation tests:
lyft_fare -- lyft_pay: r=0.849, p=6.95e-279
lyft_fare -- lyft_wait: r=0.132, p=2.75e-05
lyft_pay -- lyft_wait: r=-0.016, p=0.612

Conditional (partial) correlation tests:
lyft_fare -- lyft_pay | lyft_wait: r=0.859, p=5.88e-292
lyft_fare -- lyft_wait | lyft_pay: r=0.276, p=6.24e-19
lyft_pay -- lyft_wait | lyft_fare: r=-0.245, p=4.03e-15

Learned structure:
lyft_pay -> lyft_fare
lyft_wait -> lyft_fare
